# LLM Bias Sentinel - Google Colab Setup

This notebook sets up the entire project on Google Colab with free T4 GPU.

**Before running:** Go to `Runtime > Change runtime type > T4 GPU`

| Your Machine (CPU) | Google Colab (T4 GPU) |
|---|---|
| ~5 min per response | ~5 sec per response |
| Quick benchmark: 4-8 hours | Quick benchmark: 15-30 min |
| Full benchmark: 2-3 days | Full benchmark: 3-5 hours |

## Step 1: Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## Step 2: Install Ollama on Colab

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh
print("\nOllama installed!")

In [ ]:
# Start Ollama server in background
import subprocess
import time

# Start ollama serve in background
process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)  # Wait for server to start

# Verify it's running
!curl -s http://localhost:11434/api/tags | python3 -c "import sys,json; print('Ollama server: RUNNING')" 2>/dev/null || echo "Ollama server: STARTING..."
print(f"Ollama PID: {process.pid}")

In [ ]:
# Pull models (takes 3-5 min each on Colab's fast internet)
print("Pulling llama3 (~4.7GB)...")
!ollama pull llama3
print("\nPulling mistral (~4.4GB)...")
!ollama pull mistral
print("\nDone! Models:")
!ollama list

## Step 3: Upload Project

**Option A:** Upload ZIP (recommended)
1. Zip your project folder on your PC
2. Upload to Colab using the cell below

**Option B:** Clone from GitHub (if you pushed to GitHub)

In [ ]:
# OPTION A: Upload ZIP file
# 1. First, zip your project on your PC:
#    Right-click 'llm-bias-sentinel' folder > Send to > Compressed (zipped) folder
# 2. Then run this cell and upload the zip:

from google.colab import files
print("Click 'Choose Files' and select your llm-bias-sentinel.zip")
uploaded = files.upload()

In [ ]:
# Unzip the project
import os
zip_name = list(uploaded.keys())[0]
print(f"Unzipping {zip_name}...")
!unzip -q -o "{zip_name}" -d /content/

# Find the project directory
for d in os.listdir('/content/'):
    if 'bias-sentinel' in d.lower() or 'llm' in d.lower():
        project_dir = f'/content/{d}'
        break
else:
    # If folder name doesn't match, list what was extracted
    !ls /content/
    project_dir = '/content/llm-bias-sentinel'  # adjust if needed

os.chdir(project_dir)
print(f"Project directory: {os.getcwd()}")
!ls

In [ ]:
# OPTION B: Clone from GitHub (uncomment and edit URL)
# !git clone https://github.com/YOUR_USERNAME/llm-bias-sentinel.git
# os.chdir('/content/llm-bias-sentinel')

## Step 4: Install Dependencies

In [ ]:
# Install all project dependencies
!pip install -q langchain langchain-ollama langchain-community \
    datasets deepeval textblob pandas numpy plotly loguru pyyaml \
    python-dotenv tqdm fastapi uvicorn pydantic httpx \
    prometheus-client pytest

# Download TextBlob corpora
!python -m textblob.download_corpora lite

print("\nAll dependencies installed!")

In [ ]:
# Verify everything works
import sys
sys.path.insert(0, '.')

from src.config import config
from src.models.model_loader import load_model, generate_response

print("Config loaded OK")
print(f"Models configured: {[m['name'] for m in config.models]}")
print(f"Benchmarks: {config.benchmarks}")

# Quick test
model = load_model(provider='ollama', model_id='llama3', name='llama3-8b')
response = generate_response(model, 'Say hello in one sentence.')
print(f"\nllama3 response: {response}")
print("\n=== SETUP COMPLETE — READY TO RUN BENCHMARKS ===")

---
## Step 5: Run Benchmarks

Now run the actual evaluation pipeline!

In [ ]:
# Run quick benchmark suite (~15-30 min on T4 GPU)
!python run.py benchmarks --quick

In [ ]:
# View results
import json
results_path = 'reports/benchmark_results/all_results.json'
try:
    with open(results_path) as f:
        results = json.load(f)
    for r in results:
        print(f"\n{'='*50}")
        print(f"Model: {r.get('model', '?')} | Benchmark: {r.get('benchmark', '?')}")
        for k, v in r.items():
            if k not in ('model', 'benchmark', 'detailed_results', 'per_category', 'per_bias_type', 'per_domain', 'by_prompt_toxicity'):
                print(f"  {k}: {v}")
except FileNotFoundError:
    print("No results yet — run benchmarks first!")

## Step 6: Run Red-Team Assessment

In [ ]:
# Run quick red-team scan (~10-20 min on T4 GPU)
!python run.py red-team --quick

In [ ]:
# View red-team results
import glob
rt_files = glob.glob('reports/red_team/*.json')
if rt_files:
    with open(sorted(rt_files)[-1]) as f:
        rt = json.load(f)
    print("RED TEAM SUMMARY")
    print("="*50)
    for model, summary in rt.get('summary', {}).items():
        print(f"\nModel: {model}")
        for k, v in summary.items():
            print(f"  {k}: {v}")
else:
    print("No red-team results yet!")

## Step 7: Generate Compliance Artifacts

In [ ]:
# Generate fairness cards
!python run.py fairness-card llama3-8b
!python run.py fairness-card mistral-7b

In [ ]:
# Generate all report types
from compliance.report_templates import generate_all_reports

# Load results if they exist
bench_results = None
rt_results = None
try:
    with open('reports/benchmark_results/all_results.json') as f:
        bench_results = json.load(f)
except: pass

try:
    rt_files = sorted(glob.glob('reports/red_team/*.json'))
    if rt_files:
        with open(rt_files[-1]) as f:
            rt_results = json.load(f)
except: pass

paths = generate_all_reports('llama3-8b', bench_results, rt_results)
print("Generated reports:")
for p in paths:
    print(f"  {p}")

## Step 8: Run Tests

In [ ]:
# Run full test suite
!python -m pytest tests/ -v --tb=short 2>&1 | tail -30

## Step 9: Download Results

Download all generated reports back to your PC.

In [ ]:
# Zip all reports for download
!zip -r /content/bias_sentinel_results.zip reports/ compliance/fairness_card_*.json compliance/fairness_card_*.md

from google.colab import files
files.download('/content/bias_sentinel_results.zip')
print("\nDownload started! Save this zip and extract into your project's reports/ folder.")

---
## Summary

After running all cells above, you should have:

| Artifact | Location |
|---|---|
| Benchmark results (JSON) | `reports/benchmark_results/all_results.json` |
| Model comparison (CSV) | `reports/model_comparison_matrix.csv` |
| Red-team report (JSON) | `reports/red_team/red_team_report_*.json` |
| Red-team report (HTML) | `reports/red_team/red_team_report.html` |
| Fairness cards | `compliance/fairness_card_*.json` + `.md` |
| Executive summary | `reports/final/executive_summary_*.md` |
| Technical report | `reports/final/technical_report_*.md` |
| Compliance report | `reports/final/compliance_report_*.md` |

Download the zip and place back in your local project to complete your portfolio.